## Selección del mejor modelo

### By:
Jhonner Acosta

### Date:
2026-08-21

### Description:

Requerimiento
Realizar el proceso de selección del mejor modelo de machine learning para resolver el problema de machine Learning, este modelo debe ser mejor que el modelo baseline. Crear un nuevo branch de git (Usar Gitflow).

Se puede solucionar:

Manual
Automatizada
Seleccione la opción que considere mejor. (Es un buen momento para conocer las herramientas de AutoML)

Manual
Si lo va hacer manual Tomar como ejemplo los pasos de: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/6-model_selection/

De ser Manual este proceso se incluye :

Usar pipeline de procesamiento de datos creado en la tarea anterior
Dividir los datos en Train / Test
seleccionar multiples modelos de machine learning según el problema (mínimo 4 tipos diferentes)
Evaluar con diferentes métricas según el problema que se esta solucionando (En la tarea anterior se definió cual seria la métrica mas importante)
Eliminar los modelos que funciones por debajo del promedio de los modelos
Evaluar los modelos restantes usando validación cruzada (Cross validation) y obtener la media y la desviación estándar de la medida de las evoluciones realizadas. (recuerde usar pipelines de la union del pipeline de transformación y el modelo)
los dos o tres mejores modelos realizar el proceso de optimización de hiperparametros, recuerde usar cross validation y los pipelines
Comparar los resultados y seleccionar el mejor modelo (recuerde usar pruebas estadísticas en la comparación para seleccionar el mejor modelo)
Entrenar con todo el dataset de train el modelo seleccionado con sus hiperparametros, luego hacer la predicción con el dataset de test y realizar la evaluación de desempeño.
si los resultados de la evaluación de test, se presenta, underfitting, overfitting repetir el proceso nuevamente con otros modelos y otras configuraciones, si no se logra un buen desempeño es necesario repetir el proceso de feature engineering.
Realizar el análisis de Learning curve
obtener las gráficas de escalabilidad con tiempo de entrenamiento y score
Almacenar el pipeline de preprocesamiento + modelo
Interpretar los resultados
realizar análisis de los resultados
definir recomendaciones
Automatizada
Utilizar herramientas de AutoML para la selección del mejor modelo, por ejemplo:

AutoGlueon https://auto.gluon.ai/

Pycaret https://pycaret.org/

TabPFN https://github.com/PriorLabs/TabPFN

MLJar https://mljar.com/automl/

FLAML https://microsoft.github.io/FLAML/

Realizar el análisis de Learning curve de ser posible

Analizar Overfitting y Underfitting

obtener las gráficas de escalabilidad con tiempo de entrenamiento y score

Almacenar el pipeline de preprocesamiento + modelo

Interpretar los resultados

realizar análisis de los resultados

definir recomendaciones

Puede utilizar las librerías o herramientas que considere para resolver la tarea

Entregables
Notebook con los pasos descritos anteriormente y el modelo entrenado en formato .joblib

Se debe realizar un Pull request para ingresar el notebook a la ramamain para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD.

## Selección del mejor modelo de machine learning mediante comparación de múltiples algoritmos, optimización de hiperparámetros y validación estadística. Métrica principal: **F1-score** (contexto médico).

## 📚 Import  libraries

In [ ]:
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

_root = next(
    p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()
)
DATA_DIR = _root / "data"
MODELS_DIR = _root / "models"
MODELS_DIR.mkdir(exist_ok=True)
SEED = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

## 💾 Load data

In [2]:
df = pd.read_parquet(DATA_DIR / "02_intermediate" / "corazon_type_fixed.parquet")
df = df.drop_duplicates()

TARGET = "disease"
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# Cargar pipeline de feature engineering
full_pipeline = joblib.load(MODELS_DIR / "feature_pipeline.pkl")

# Preparar X compatible con el pipeline
cat_cols = ["sex", "chest_pain", "fbs", "rest_ecg", "slope", "ca", "thal"]
num_cols = ["age", "rest_bp", "chol", "max_hr", "old_peak"]

X_clean = X.drop(columns=["exang"], errors="ignore").copy()
for col in cat_cols:
    X_clean[col] = X_clean[col].astype(object).where(X_clean[col].notna(), other=np.nan)
for col in num_cols:
    X_clean[col] = pd.to_numeric(X_clean[col], errors="coerce").astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Distribución target train:\n{y_train.value_counts()}")

Train: (323, 12), Test: (81, 12)
Distribución target train:
disease
0    179
1    144
Name: count, dtype: int64


## 👷 Comparacion inicial de modelos


In [3]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "SVM": SVC(probability=True, random_state=SEED),
}

results = {}
for name, model in models.items():
    pipe = Pipeline(
        [
            ("preprocessor", full_pipeline.named_steps["preprocessor"]),
            ("classifier", model),
        ]
    )
    scores = cross_val_score(pipe, X_clean, y, cv=CV_FOLDS, scoring="f1")
    results[name] = scores
    print(f"{name:25s}: F1 = {scores.mean():.4f} ± {scores.std():.4f}")

Logistic Regression      : F1 = 0.8185 ± 0.0666
Random Forest            : F1 = 0.8326 ± 0.0752
Gradient Boosting        : F1 = 0.7894 ± 0.0957
SVM                      : F1 = 0.8220 ± 0.0635


/home/jhonner/Heart-project-new/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/jhonner/Heart-project-new/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/jhonner/Heart-project-new/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/jhonner/Heart-project-new/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarn

## Eliminar modelos por debajo del promedio

In [4]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC as SVCBase

# Reconstruir models con SVM corregido
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "SVM": CalibratedClassifierCV(SVCBase(random_state=SEED), ensemble=False),
}

mean_scores = {name: scores.mean() for name, scores in results.items()}
overall_mean = np.mean(list(mean_scores.values()))
print(f"Media global F1: {overall_mean:.4f}")
print()

selected_models = {}
for name, mean in mean_scores.items():
    status = "✓ SELECCIONADO" if mean >= overall_mean else "✗ eliminado"
    print(f"{name:25s}: {mean:.4f} {status}")
    if mean >= overall_mean:
        selected_models[name] = models[name]

Media global F1: 0.8156

Logistic Regression      : 0.8185 ✓ SELECCIONADO
Random Forest            : 0.8326 ✓ SELECCIONADO
Gradient Boosting        : 0.7894 ✗ eliminado
SVM                      : 0.8220 ✓ SELECCIONADO


## Optimización de hiperparámetros

In [5]:
param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10],
        "classifier__solver": ["lbfgs", "liblinear"],
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 5, 10],
        "classifier__min_samples_split": [2, 5],
    },
    "SVM": {
        "classifier__estimator__C": [0.1, 1, 10],
        "classifier__estimator__kernel": ["rbf", "linear"],
    },
}

best_models = {}
for name, model in selected_models.items():
    pipe = Pipeline(
        [
            ("preprocessor", full_pipeline.named_steps["preprocessor"]),
            ("classifier", model),
        ]
    )
    grid = GridSearchCV(pipe, param_grids[name], cv=CV_FOLDS, scoring="f1", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    print(f"{name}")
    print(f"  Mejores params: {grid.best_params_}")
    print(f"  Mejor F1 CV:   {grid.best_score_:.4f}")
    print()

Logistic Regression
  Mejores params: {'classifier__C': 0.1, 'classifier__solver': 'lbfgs'}
  Mejor F1 CV:   0.7942

Random Forest
  Mejores params: {'classifier__max_depth': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
  Mejor F1 CV:   0.8197

SVM
  Mejores params: {'classifier__estimator__C': 1, 'classifier__estimator__kernel': 'rbf'}
  Mejor F1 CV:   0.8231



## 📊 Analysis of Results and Conclusions 

Description of the results obtained and if there are conclusions that can be drawn from them.

The analysis of results must be related to the description of the task.

**Note:** An analysis of results does not necessarily lead to conclusions, but to ideas or proposals for future work


## Comparación estadística con prueba de Wilcoxon


In [12]:
# Obtener scores CV de los modelos optimizados
ALPHA = 0.05
optimized_scores = {}
for name, model in best_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring="f1")
    optimized_scores[name] = scores
    print(f"{name:25s}: F1 = {scores.mean():.4f} ± {scores.std():.4f}")

print("\n--- Prueba de Wilcoxon (comparación par a par) ---")
names = list(optimized_scores.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        stat, p = stats.wilcoxon(optimized_scores[a], optimized_scores[b])
        sig = "significativo" if p < ALPHA else "no significativo"
        print(f"{a} vs {b}: p={p:.4f} ({sig})")

Logistic Regression      : F1 = 0.7942 ± 0.0308
Random Forest            : F1 = 0.8197 ± 0.0281
SVM                      : F1 = 0.8231 ± 0.0325

--- Prueba de Wilcoxon (comparación par a par) ---
Logistic Regression vs Random Forest: p=0.2500 (no significativo)
Logistic Regression vs SVM: p=0.3125 (no significativo)
Random Forest vs SVM: p=0.8750 (no significativo)


## Selección y evaluación final en Test

Las pruebas de Wilcoxon no muestran diferencias significativas entre los modelos (dataset pequeño, baja potencia estadística con 5 folds). Se selecciona **SVM** por tener el mayor F1 promedio (0.8231).

In [8]:
best_model_name = "SVM"
final_model = best_models[best_model_name]

# Reentrenar con todo el train set
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print(f"=== Evaluación en Test — {best_model_name} ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"\n{classification_report(y_test, y_pred)}")

=== Evaluación en Test — SVM ===
Accuracy:  0.8642
F1 Score:  0.8571
AUC-ROC:   0.9296

              precision    recall  f1-score   support

           0       0.93      0.82      0.87        45
           1       0.80      0.92      0.86        36

    accuracy                           0.86        81
   macro avg       0.86      0.87      0.86        81
weighted avg       0.87      0.86      0.86        81



## Matriz de confusión

In [9]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusión — {best_model_name}")
plt.tight_layout()
plt.savefig("confusion_matrix_svm.png", dpi=72)
plt.show()

/tmp/ipykernel_18399/492151362.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Learning Curve y Escalabilidad

In [10]:
train_sizes, train_scores, test_scores, fit_times, score_times = learning_curve(
    final_model,
    X_clean,
    y,
    cv=CV_FOLDS,
    scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 10),
    return_times=True,
    random_state=SEED,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_sizes, train_scores.mean(axis=1), "o-", label="Train")
axes[0].plot(train_sizes, test_scores.mean(axis=1), "o-", label="Validation")
axes[0].fill_between(
    train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.2,
)
axes[0].fill_between(
    train_sizes,
    test_scores.mean(axis=1) - test_scores.std(axis=1),
    test_scores.mean(axis=1) + test_scores.std(axis=1),
    alpha=0.2,
)
axes[0].set_title("Learning Curve — SVM")
axes[0].set_xlabel("Training examples")
axes[0].set_ylabel("F1 Score")
axes[0].legend()

axes[1].plot(train_sizes, fit_times.mean(axis=1), "o-")
axes[1].set_title("Escalabilidad — Tiempo de entrenamiento")
axes[1].set_xlabel("Training examples")
axes[1].set_ylabel("Tiempo (s)")

axes[2].plot(fit_times.mean(axis=1), test_scores.mean(axis=1), "o-")
axes[2].set_title("Score vs Tiempo de entrenamiento")
axes[2].set_xlabel("Tiempo (s)")
axes[2].set_ylabel("F1 Score")

plt.tight_layout()
plt.savefig("learning_curve_svm.png", dpi=72)
plt.show()

/tmp/ipykernel_18399/1670514416.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# ── 19. Guardar pipeline final ──────────────────────────────────────────────
model_path = MODELS_DIR / "06_svm_model.joblib"
joblib.dump(final_model, model_path)
print(f"Modelo guardado en: {model_path.relative_to(_root)}")

Modelo guardado en: models/06_svm_model.joblib


## Interpretación del modelo seleccionado

El modelo final seleccionado es **SVM (Support Vector Machine)** envuelto en `CalibratedClassifierCV`, integrado dentro del pipeline de feature engineering previamente definido.

### ¿Por qué SVM?

| Criterio | Valor |
|---|---|
| F1-Score (CV) | 0.8231 |
| F1-Score (test) | 0.8571 |
| AUC-ROC (test) | 0.9296 |

- **Discriminación excelente**: AUC-ROC de 0.93 indica que el modelo distingue bien entre pacientes con y sin enfermedad cardíaca.
- **Balance precisión-recall**: F1 de 0.86 refleja un equilibrio adecuado, crucial en un contexto médico donde tanto falsos positivos como falsos negativos tienen consecuencias.
- **Estabilidad**: Las curvas de aprendizaje muestran convergencia sin sobreajuste severo. Con ~250 muestras de entrenamiento el modelo alcanza su rendimiento máximo.
- **Escalabilidad**: El tiempo de entrenamiento es bajo (< 1s), adecuado para datasets de este tamaño.

### Limitaciones
- Dataset pequeño (404 registros) → los intervalos de confianza son amplios.
- La prueba de Wilcoxon no detectó diferencias estadísticamente significativas entre los top 3 modelos, por lo que la elección de SVM se basa en la métrica F1 observada.
- En producción se recomienda monitorear el modelo ante nuevos datos y re-entrenar periódicamente.

## 💡 Proposals and Ideas

From the results obtained, what ideas or proposals can be generated to continue with the project


## 9. Conclusiones y Recomendaciones

### Conclusiones
1. **Cuatro modelos evaluados**: Regresión Logística, Random Forest, Gradient Boosting y SVM. GB fue eliminado por rendimiento inferior al promedio global (F1=0.7894 < 0.8156).
2. **Optimización de hiperparámetros** vía GridSearchCV con 5-fold CV: SVM obtuvo F1=0.8231, superior a LR (0.7942) y RF (0.8197).
3. **Comparación estadística** (Wilcoxon signed-rank test): no se encontraron diferencias significativas (p>0.05), lo que es esperable en datasets pequeños. Se seleccionó SVM como mejor modelo por métrica observada.
4. **Evaluación en test**: Accuracy=0.8642, F1=0.8571, AUC-ROC=0.9296 — resultados consistentes con la validación cruzada, sin señales de sobreajuste.
5. **Pipeline integrado** (feature engineering + modelo) guardado en `models/06_svm_model.joblib` listo para inferencia.

### Recomendaciones
- **Ampliar el dataset**: Recolectar más registros para reducir varianza en las métricas y aumentar el poder estadístico de las pruebas de comparación.
- **Calibración de probabilidades**: El uso de `CalibratedClassifierCV` ya provee probabilidades calibradas; validar con reliability diagrams antes de producción.
- **Umbral de decisión**: Evaluar si el umbral por defecto (0.5) es óptimo para el contexto clínico, ajustando según la tolerancia a falsos negativos (pacientes enfermos clasificados como sanos).
- **Monitoreo en producción**: Implementar alertas ante data drift y programar re-entrenamiento trimestral.